# Practica 1 - Saturacion del uplink en un campus

Este notebook estima la carga uplink de una celula WCDMA/UMTS en hora punta y calcula el numero maximo de usuarios simultaneos antes de alcanzar un umbral operativo de carga.

Se comparan dos escenarios:
- Control de potencia eficaz.
- Control de potencia degradado por movilidad alta o calibracion imperfecta.

El analisis usa perfiles de 12.2 kb/s, 32 kb/s y 64 kb/s, velocidades de 3, 15 y 50 km/h, factor de actividad entre 0.4 y 0.7, interferencia intercelular parametrizable y un umbral objetivo de carga de 0.70.

## Modelo simplificado

La carga uplink por usuario se aproxima con:

$$\eta_u = (1 + i) \cdot \frac{1}{1 + \frac{W}{(E_b/N_0)_{req} \cdot R \cdot v}}$$

Donde W es el ancho de banda de chip, R es la tasa binaria del servicio, v es el factor de actividad e i representa la interferencia intercelular relativa.

Para un conjunto de usuarios homogeneo:

$$N_{max} = \left\lfloor \frac{\eta_{objetivo}}{\eta_u} \right\rfloor$$

En un escenario mixto, la carga media por usuario se obtiene ponderando cada perfil por su proporcion dentro de la mezcla de trafico.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

In [ ]:
W_HZ = 3.84e6
TARGET_LOAD = 0.70
INTERCELL_FACTOR = 0.65
SPEEDS_KMH = [3, 15, 50]
ACTIVITY_GRID = np.round(np.arange(0.40, 0.71, 0.05), 2)

service_profiles = pd.DataFrame(
    [
        {"service": "Audio 12.2 kb/s", "rate_kbps": 12.2, "default_activity": 0.45, "ebn0_db": 5.0, "share": 0.45},
        {"service": "Subida media 32 kb/s", "rate_kbps": 32.0, "default_activity": 0.55, "ebn0_db": 4.5, "share": 0.35},
        {"service": "Subida intensiva 64 kb/s", "rate_kbps": 64.0, "default_activity": 0.65, "ebn0_db": 4.0, "share": 0.20},
    ]
)

power_control_scenarios = {
    "control_eficaz": {
        "mobility_margin_db": {3: 0.5, 15: 1.0, 50: 1.8},
        "calibration_margin_db": 0.2,
    },
    "control_degradado": {
        "mobility_margin_db": {3: 1.5, 15: 2.8, 50: 4.5},
        "calibration_margin_db": 1.0,
    },
}

service_profiles

In [ ]:
def effective_ebn0_linear(base_ebn0_db, speed_kmh, scenario_name):
    scenario = power_control_scenarios[scenario_name]
    total_margin_db = scenario['mobility_margin_db'][speed_kmh] + scenario['calibration_margin_db']
    return 10 ** ((base_ebn0_db + total_margin_db) / 10.0)


def user_load(rate_kbps, activity_factor, base_ebn0_db, speed_kmh, scenario_name, intercell_factor=INTERCELL_FACTOR):
    rate_bps = rate_kbps * 1e3
    ebn0_linear = effective_ebn0_linear(base_ebn0_db, speed_kmh, scenario_name)
    denominator = 1.0 + W_HZ / (ebn0_linear * rate_bps * activity_factor)
    return (1.0 + intercell_factor) / denominator


def max_users_for_service(rate_kbps, activity_factor, base_ebn0_db, speed_kmh, scenario_name, target_load=TARGET_LOAD):
    load = user_load(rate_kbps, activity_factor, base_ebn0_db, speed_kmh, scenario_name)
    return int(np.floor(target_load / load)), load


def max_users_for_mix(activity_factor, speed_kmh, scenario_name, target_load=TARGET_LOAD):
    mean_load = 0.0
    for row in service_profiles.itertuples(index=False):
        mean_load += row.share * user_load(row.rate_kbps, activity_factor, row.ebn0_db, speed_kmh, scenario_name)
    return int(np.floor(target_load / mean_load)), mean_load


def admission_decision(current_load, service_name, speed_kmh, scenario_name, activity_factor, threshold=0.72):
    row = service_profiles.loc[service_profiles['service'] == service_name].iloc[0]
    projected_load = current_load + user_load(row.rate_kbps, activity_factor, row.ebn0_db, speed_kmh, scenario_name)
    admit = projected_load <= threshold
    return projected_load, admit

In [ ]:
service_rows = []
for scenario_name in power_control_scenarios:
    for speed_kmh in SPEEDS_KMH:
        for row in service_profiles.itertuples(index=False):
            max_users, load_per_user = max_users_for_service(
                row.rate_kbps,
                row.default_activity,
                row.ebn0_db,
                speed_kmh,
                scenario_name,
            )
            service_rows.append(
                {
                    'scenario': scenario_name,
                    'speed_kmh': speed_kmh,
                    'service': row.service,
                    'activity_factor': row.default_activity,
                    'load_per_user': load_per_user,
                    'n_max': max_users,
                }
            )

service_capacity = pd.DataFrame(service_rows)
service_capacity.pivot_table(index=['service', 'activity_factor'], columns=['scenario', 'speed_kmh'], values='n_max')

In [ ]:
mix_rows = []
for scenario_name in power_control_scenarios:
    for speed_kmh in SPEEDS_KMH:
        for activity_factor in ACTIVITY_GRID:
            max_users, mean_load = max_users_for_mix(activity_factor, speed_kmh, scenario_name)
            mix_rows.append(
                {
                    'scenario': scenario_name,
                    'speed_kmh': speed_kmh,
                    'activity_factor': activity_factor,
                    'mean_load_per_user': mean_load,
                    'n_max': max_users,
                }
            )

mix_capacity = pd.DataFrame(mix_rows)
summary_mix = mix_capacity[mix_capacity['activity_factor'].isin([0.40, 0.55, 0.70])]
summary_mix.pivot_table(index='activity_factor', columns=['scenario', 'speed_kmh'], values='n_max')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
selected_activities = [0.40, 0.55, 0.70]
scenario_titles = {
    'control_eficaz': 'Control de potencia eficaz',
    'control_degradado': 'Control de potencia degradado',
}

for axis, scenario_name in zip(axes, power_control_scenarios):
    subset = mix_capacity[mix_capacity['scenario'] == scenario_name]
    for activity_factor in selected_activities:
        curve = subset[subset['activity_factor'] == activity_factor].sort_values('speed_kmh')
        axis.plot(curve['speed_kmh'], curve['n_max'], marker='o', linewidth=2, label=f'v={activity_factor:.2f}')
    axis.set_title(scenario_titles[scenario_name])
    axis.set_xlabel('Velocidad [km/h]')
    axis.set_ylabel('N maximo en mezcla')
    axis.set_xticks(SPEEDS_KMH)
    axis.legend(title='Actividad')

fig.suptitle('Capacidad maxima frente a la velocidad')
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for axis, scenario_name in zip(axes, power_control_scenarios):
    subset = mix_capacity[mix_capacity['scenario'] == scenario_name]
    for speed_kmh in SPEEDS_KMH:
        curve = subset[subset['speed_kmh'] == speed_kmh].sort_values('activity_factor')
        axis.plot(curve['activity_factor'], curve['n_max'], marker='o', linewidth=2, label=f'{speed_kmh} km/h')
    axis.set_title(scenario_titles[scenario_name])
    axis.set_xlabel('Factor de actividad')
    axis.set_ylabel('N maximo en mezcla')
    axis.legend(title='Velocidad')

fig.suptitle('Capacidad maxima frente al factor de actividad')
fig.tight_layout()
plt.show()

In [ ]:
stress_case = []
for scenario_name in power_control_scenarios:
    current_load = 0.68 if scenario_name == 'control_eficaz' else 0.66
    for service_name in service_profiles['service']:
        projected_load, admit = admission_decision(
            current_load=current_load,
            service_name=service_name,
            speed_kmh=15,
            scenario_name=scenario_name,
            activity_factor=0.60,
            threshold=0.72,
        )
        stress_case.append(
            {
                'scenario': scenario_name,
                'current_load': current_load,
                'service': service_name,
                'projected_load': projected_load,
                'admit_session': admit,
            }
        )

admission_table = pd.DataFrame(stress_case)
admission_table

## Lectura tecnica de resultados

1. La subida de 64 kb/s consume mucha mas carga por usuario porque reduce la ganancia de procesado y cae antes la capacidad total.
2. A mayor velocidad, el margen extra de potencia para sostener la calidad aumenta y el numero maximo de usuarios disminuye.
3. El control de potencia degradado penaliza mas el uplink porque todos los terminales comparten la misma reserva de interferencia.
4. Una regla de admision simple y robusta es congelar nuevas sesiones de subida intensiva cuando la carga instantanea supera 0.70 y bloquearlas de forma estricta a partir de 0.72.
5. Esta regla puede complementarse con reintentos diferidos para sincronizacion en la nube o cargas no urgentes del LMS.